This vignette demonstrates how to use the bioc2ri ecosystem to seamlessly move genomic and single-cell data between Python and R.

## Environment Setup
First, we initialize our conversion engines. In this architecture, plugins are designed to be modular. While you can combine them, the bioc_experiment_plugin is smart enough to call the bioc_ranges_plugin and biocpy_plugin internally as needed.

In [1]:
import numpy as np
from biocframe import BiocFrame
from iranges import IRanges
from genomicranges import GenomicRanges
from singlecellexperiment import SingleCellExperiment

# Import our plugins
from bioc2ri import base_plugin
from bioc2ri.bioc_ranges_plugin import bioc_ranges_plugin
from bioc2ri.summarizedexperiment_plugin import bioc_experiment_plugin

# Initialize the standalone experiment engine
# It will internally resolve ranges and frames.
exp_engine = bioc_experiment_plugin()
ranges_engine = bioc_ranges_plugin()

## Constructing Genomic Ranges
We'll start by creating a GenomicRanges object in Python. This represents the genomic coordinates of our features (e.g., genes or peaks).

In [2]:
# Create 5 genomic features
gr = GenomicRanges(
    seqnames=["chr1", "chr1", "chr2", "chrX", "chrY"],
    ranges=IRanges(start=[100, 200, 300, 400, 500], width=[50, 50, 50, 50, 50]),
    strand=["+", "-", "+", "*", "-"],
    mcols=BiocFrame({"gene_id": ["G1", "G2", "G3", "G4", "G5"]})
)

# Python -> R Conversion
r_gr = ranges_engine.py2r(gr)
print(f"R GRanges type: {type(r_gr)}")

R GRanges type: <class 'rpy2.robjects.methods.RS4'>


In [3]:
from genomicranges import GenomicRanges, CompressedGenomicRangesList
from compressed_lists import Partitioning
from rpy2.robjects import r
    
# 1. In Python: One Gene, Two Exons
exon1 = GenomicRanges(seqnames=["chr1"], ranges=IRanges([100], [50]), strand=["+"])
exon2 = GenomicRanges(seqnames=["chr1"], ranges=IRanges([200], [50]), strand=["+"])

grl = CompressedGenomicRangesList.from_list(
    lst=[exon1, exon2], 
    names=["Gene_A", "Gene_B"] # For simplicity, each gene has 1 exon here
)

# 2. Convert to R
# Internal: Converts to a single flat GRanges + Partitioning ends [1, 2]
r_grl = ranges_engine.py2r(grl)

# 3. Verify in R
print(r["class"](r_grl)) # 'CompressedGRangesList'
print(r["elementNROWS"](r_grl)) # 1 1

# 4. Round-trip
grl_back = ranges_engine.r2py(r_grl)
assert isinstance(grl_back, CompressedGenomicRangesList)

[1] "CompressedGRangesList"
attr(,"package")
[1] "GenomicRanges"

Gene_A Gene_B 
     1      1 



## Building a SingleCellExperiment
Now, let's wrap those ranges into a SingleCellExperiment (SCE). This is the "boss" of Bioconductor objects, containing assays (matrices), row/column metadata, and reduced dimensions.

In [4]:
# 1. Setup Assays (5 genes x 4 cells)
counts = np.random.poisson(lam=5, size=(5, 4))
logcounts = np.log1p(counts)

# 2. Setup Column Metadata (Samples)
coldata = BiocFrame({
    "sample": ["S1", "S1", "S2", "S2"],
    "batch": ["A", "A", "B", "B"]
}, row_names=["Cell_1", "Cell_2", "Cell_3", "Cell_4"])

# 3. Create the SCE
sce = SingleCellExperiment(
    assays={"counts": counts, "logcounts": logcounts},
    row_ranges=gr, # Using the GRanges we made above
    column_data=coldata,
    reduced_dims={"PCA": np.random.normal(size=(4, 2))}
)

# Python -> R Conversion
r_sce = exp_engine.py2r(sce)

In [5]:
r["print"](r_sce)

class: SingleCellExperiment 
dim: 5 4 
metadata(0):
assays(2): counts logcounts
rownames: NULL
rowData names(1): gene_id
colnames(4): Cell_1 Cell_2 Cell_3 Cell_4
colData names(2): sample batch
reducedDimNames(1): PCA
mainExpName: NULL
altExpNames(0):


<rpy2.robjects.methods.RS4 object at 0x7ce8ed59aed0> [25]
R classes: ('SingleCellExperiment',)

## Verification in R
Using rpy2, we can verify that the R-side SingleCellExperiment is fully valid and parallel.

In [6]:
from rpy2.robjects import r

# Check R class
print(r["class"](r_sce)) # Should show ['SingleCellExperiment', 'RangedSummarizedExperiment', ...]

# Check dimensions
print(f"R Dimensions: {r['dim'](r_sce)}")

# Check if rowRanges transitioned correctly
print(f"R Strand info: {r['strand'](r['rowRanges'](r_sce))}")

[1] "SingleCellExperiment"
attr(,"package")
[1] "SingleCellExperiment"

R Dimensions: [1] 5 4

R Strand info: factor-Rle of length 5 with 5 runs
  Lengths: 1 1 1 1 1
  Values : - * - + *
Levels(3): + - *



## Round-trip: R to Python
The true test of a plugin is the ability to bring data back without loss of information.

In [7]:
# R -> Python Conversion
sce_back = exp_engine.r2py(r_sce)

# Verify identity
assert (sce_back.get_assay("counts") == counts).all()
assert sce_back.get_column_names() == sce.get_column_names()
assert sce_back.get_row_ranges().get_seqnames() == sce.get_row_ranges().get_seqnames()

print("Round-trip successful! All slots synchronized.")

Round-trip successful! All slots synchronized.
